# Dear Canada — Place Matching (pipeline 2 of 2)

Attaches the place names found by `01_place_extraction.ipynb` to geographies. Matching is focused on Canada and on historical names:

1. **Overrides**: hand-curated historical names (Upper Canada, Bytown, Ville-Marie, Grosse Île…), bounded by year and optionally by book.
2. **CGNDB**: the Canadian Geographical Names Database, filtered by the place type suggested by the extraction step (city vs province, river vs settlement) and ranked by prominence and by the provinces each book is set in.
3. **CHGIS**: census subdivisions of the Canadian Historical GIS (1851–1921), as a fallback for names that only exist in historical census geography, and to attach each matched Canadian place to its census subdivision.
4. **World list**: a short curated list of countries, regions and cities outside Canada, matched by exact name only. There is no worldwide fuzzy search, so a character's name can no longer land in Poland or North Carolina.

Each mention gets the historical political unit of its year (New France, Lower Canada, Canada East, Quebec…).

Status: first version of the split pipeline. Weights and thresholds are in the configuration cell and should be tuned against the review output (section 9).

## 0. Requirements, inputs and outputs

### Environment
- Python 3.10 or later, run locally in Jupyter.
- Packages: `pandas`, `numpy`, `geopandas` (1.0 or later), `shapely` (2.0 or later), `pyogrio`, `pyproj`. `tqdm` optional.
- `pyogrio` reads the CHGIS file geodatabase through GDAL's OpenFileGDB driver. No ESRI software is required.

### Network access
None. All gazetteers are local files.

### Gazetteer data to make available

| Gazetteer | Put it in | How to get it |
|---|---|---|
| CGNDB | `data/Gazetteers/cgndb/` | National CSV, English: `cgn_canada_csv_eng.zip` from `https://ftp.cartes.canada.ca/pub/nrcan_rncan/vector/geobase_cgn_toponyme/prov_csv_eng/`. Unzip; provincial files also work. Regenerated weekly, so record the download date in `data/Gazetteers/README.md` |
| CHGIS | `data/Gazetteers/chgis/` | The census subdivision file geodatabase (a `.gdb` folder, one layer per census year 1851–1921). Optional: without it, the CHGIS steps are skipped with a notice |
| Overrides | `data/Gazetteers/curated/place_overrides.csv` | Hand-edited (schema below) |
| World list | `data/Gazetteers/curated/world_places.csv` | Hand-edited (schema below) |

### Data the notebook reads
| Item | Path variable | Notes |
|---|---|---|
| Extracted mentions | `EXTRACTED_CSV` | Output of `01_place_extraction.ipynb`. Only rows with `keep = True` are used |
| Gazetteers | `CGNDB_DIR`, `CHGIS_DIR`, `OVERRIDES_CSV`, `WORLD_CSV` | A missing gazetteer is skipped with a notice |

Overrides CSV (`name`, `latitude`, `longitude` required):

| Column | Description |
|---|---|
| `name`, `aliases` | Name as written, and `\|`-separated alternatives (matched after folding) |
| `place_type` | Type the row applies to; blank = any |
| `book` | Book code; blank = all books |
| `year_from`, `year_to` | Entry-year range; blank = open |
| `latitude`, `longitude` | WGS84 representative point |
| `matched_name` | Label reported in the output |
| `province` | Modern province or territory, lower case |
| `feature_class` | One of the classes in section 3.1 |
| `location_precision` | `site`, `settlement`, `region` or `country` |
| `note` | Free text, not used |

World list CSV: `name`, `aliases`, `place_type` (`country`, `region`, `settlement`, `water`, `landform`), `country`, `region`, `latitude`, `longitude`, `note`.

### Outputs

Final outputs (`OUTPUT_DIR` = `data/Pipe Outputs/places/`):

| File | Content |
|---|---|
| `place_mentions.csv` | Main result. One row per kept mention, with its book context and location. Published to the website as the places dataset |
| `place_matches.csv` | One row per place query (a name of one type within one book) with the match details. Also published |
| `place_matches.geojson` | Point features for located queries |
| `csd_polygons.geojson` | Census subdivision polygons linked at least once (needs CHGIS) |

Intermediate outputs (`Pipelines/Place Mentions/intermediate/matching/`): `place_candidates.csv` (top candidates per query, for review), `book_context.csv` (province shares per book, section 5), `run_metadata.json`.

Before writing, the previous final outputs are copied to `data/Pipe Outputs/Archived Versions/places/<timestamp>/` (the newest `ARCHIVE_KEEP` copies are kept).

Schema of `place_mentions.csv`:

| Column | Description |
|---|---|
| `mention_id`, `book`, `chapter_id`, `entry_id` | From the extraction step |
| `date`, `date_precision`, `year_inferred`, `date_raw` | Entry date (ISO 8601) and its precision |
| `place`, `place_type`, `type_confidence`, `type_evidence` | Name as cleaned, and the type suggested by the extraction step |
| `ner_label`, `char_start`, `char_end`, `sentence` | Model label and position in the entry |
| `historicity` | True if the mention has coordinates. Kept under this name for the website; it means *located*, not *verified historical* |
| `resolution_status` | `override`, `resolved_ca` (CGNDB or CHGIS), `resolved_world`, `unresolved`, `not_geographic` (vessels) |
| `gazetteer`, `matched_name`, `feature_type`, `location_precision` | Source and nature of the match. `location_precision` is `site`, `settlement`, `feature`, `region` or `country` |
| `region_historical` | Political unit at the entry's year (section 7) |
| `region_modern`, `country` | Modern province, territory or state, and country |
| `latitude`, `longitude` | WGS84 |
| `csd_name`, `csd_layer_year` | Census subdivision, if linked |
| `needs_review`, `review_reasons` | See `place_matches.csv` |
| `query_id` | Join key to `place_matches.csv` |

### Known limitations
- CGNDB records current official names and modern positions; a match is a modern feature with the same name, not a verified historical place. Overrides and CHGIS cover the historical names known so far.
- Story locations are in no gazetteer and stay `unresolved`.
- Book context (section 5) is computed from the places the book mentions most; a book whose most frequent places are themselves ambiguous gets weak context. `BOOK_PROVINCE_HINTS` lets you set it by hand.
- CHGIS covers census years 1851–1921. Books outside that range (within `CHGIS_MAX_GAP_YEARS`) get no census subdivision.
- The CGNDB carries a content advisory: it contains historical terminology considered racist, offensive or derogatory. Names are matched and reproduced as published.

### Paths

In [1]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Every file system location used by the notebook is defined here.
from pathlib import Path

def find_project_root(start=None):
    """The 'Dear Canada' folder: the first parent that contains data/Book Repository."""
    start = Path(start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / "data" / "Book Repository").is_dir():
            return p
    raise FileNotFoundError(f"No parent of {start} contains data/Book Repository")

PROJECT_ROOT = find_project_root()
PIPELINE_DIR = PROJECT_ROOT / "Pipelines" / "Place Mentions"

# Input: output of 01_place_extraction.ipynb
EXTRACTED_CSV = PIPELINE_DIR / "intermediate" / "place_mentions_extracted.csv"

# Gazetteers
GAZETTEER_DIR = PROJECT_ROOT / "data" / "Gazetteers"
CGNDB_DIR     = GAZETTEER_DIR / "cgndb"               # national or provincial CGNDB CSV files
CGNDB_GLOB    = "*.csv"
CHGIS_DIR     = GAZETTEER_DIR / "chgis"               # the first .gdb (or .gpkg) found here is used
CHGIS_GLOBS   = ["*.gdb", "*.gpkg"]
OVERRIDES_CSV = GAZETTEER_DIR / "curated" / "place_overrides.csv"
WORLD_CSV     = GAZETTEER_DIR / "curated" / "world_places.csv"
CORRECTIONS_CSV = GAZETTEER_DIR / "curated" / "place_corrections.csv"  # flags made on the website

# Intermediate outputs (kept in the pipeline folder)
WORK_DIR          = PIPELINE_DIR / "intermediate" / "matching"
CANDIDATES_CSV    = WORK_DIR / "place_candidates.csv"
BOOK_CONTEXT_CSV  = WORK_DIR / "book_context.csv"
RUN_METADATA_JSON = WORK_DIR / "run_metadata.json"

# Final outputs
OUTPUT_DIR      = PROJECT_ROOT / "data" / "Pipe Outputs" / "places"
MENTIONS_CSV    = OUTPUT_DIR / "place_mentions.csv"
MATCHES_CSV     = OUTPUT_DIR / "place_matches.csv"
MATCHES_GEOJSON = OUTPUT_DIR / "place_matches.geojson"
CSD_GEOJSON     = OUTPUT_DIR / "csd_polygons.geojson"
ARCHIVE_DIR     = PROJECT_ROOT / "data" / "Pipe Outputs" / "Archived Versions" / "places"

# Website
WEB_DIR          = PROJECT_ROOT / "Web"
WEB_SOURCES_JSON = WEB_DIR / "data_sources.json"
WEB_SYNC_SCRIPT  = WEB_DIR / "scripts" / "sync_data.py"

CHGIS_PATH = next((p for g in CHGIS_GLOBS for p in sorted(CHGIS_DIR.glob(g))), None)

for d in (WORK_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)
for label, p in [("Mentions", EXTRACTED_CSV), ("CGNDB folder", CGNDB_DIR), ("CHGIS", CHGIS_PATH or CHGIS_DIR),
                 ("Overrides", OVERRIDES_CSV), ("World list", WORLD_CSV), ("Corrections", CORRECTIONS_CSV), ("Web sources", WEB_SOURCES_JSON)]:
    print(f"{label:13}: {p}  {'(exists)' if p and Path(p).exists() and (label != 'CHGIS' or CHGIS_PATH) else '(NOT FOUND)'}")
print(f"{'Output':13}: {OUTPUT_DIR}")

Mentions     : /Users/sophiebird/Downloads/Dear Canada/Pipelines/Place Mentions/intermediate/place_mentions_extracted.csv  (exists)
CGNDB folder : /Users/sophiebird/Downloads/Dear Canada/data/Gazetteers/cgndb  (exists)
CHGIS        : /Users/sophiebird/Downloads/Dear Canada/data/Gazetteers/chgis  (NOT FOUND)
Overrides    : /Users/sophiebird/Downloads/Dear Canada/data/Gazetteers/curated/place_overrides.csv  (exists)
World list   : /Users/sophiebird/Downloads/Dear Canada/data/Gazetteers/curated/world_places.csv  (exists)
Web sources  : /Users/sophiebird/Downloads/Dear Canada/Web/data_sources.json  (exists)
Output       : /Users/sophiebird/Downloads/Dear Canada/data/Pipe Outputs/places


### Configuration

In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────
BOOKS = None                       # None = all books; or a list of codes, e.g. ["SofS"]

# CHGIS
CHGIS_MAX_GAP_YEARS = 10           # maximum distance between a place's year and the census layer used
CHGIS_CENSUS_RANGE  = (1851, 1921)

# Name matching
MIN_NAME_SCORE    = 0.88           # minimum name similarity to accept a candidate
FUZZY_ENABLED     = True
FUZZY_MIN_LEN     = 6              # names shorter than this are matched exactly only
FUZZY_CONFIDENCES = {"model", "high", "medium"}   # fuzzy matching only when the type is reasonably sure
ARTICLE_PENALTY   = 0.02           # match made after removing a leading "the"/"a"/"an"
VARIANT_PENALTY   = 0.04           # match made after a rule in VARIANT_LEADING / VARIANT_TRAILING
VARIANT_LEADING   = ["city of", "town of", "village of"]
VARIANT_TRAILING  = ["city", "town", "village"]
TYPE_VARIANTS     = {              # generic words added for a type when the name lacks them ("St. Lawrence" -> "St. Lawrence River")
    "water":    {"prefix": ["lake"], "suffix": ["river"]},
    "landform": {"prefix": [], "suffix": ["island"]},
}
MAX_RAW_CANDIDATES = 200           # per gazetteer and query, before scoring

# Type compatibility: for each suggested place type, the feature classes (section 3.1) a match may have,
# with a bonus added to the score. Classes not listed are rejected. For queries with type_confidence "low",
# the classes of "unknown" are allowed as well.
TYPE_CLASSES = {
    "settlement": {"settlement": 0.10, "csd": 0.08, "area": 0.00, "facility": 0.00},
    "region":     {"admin_large": 0.10, "area": 0.05, "csd": 0.02, "settlement": -0.05},
    "country":    {"country": 0.10, "admin_large": 0.00},
    "water":      {"water": 0.10, "settlement": -0.08},   # towns named after water: Duck Lake, Owen Sound
    "landform":   {"island": 0.10, "terrain": 0.10, "area": 0.00, "settlement": -0.05},
    "facility":   {"facility": 0.10, "settlement": 0.00, "csd": 0.00, "area": -0.05},
    "unknown":    {"settlement": 0.05, "admin_large": 0.05, "country": 0.05, "csd": 0.03, "area": 0.00,
                   "island": 0.00, "water": -0.05, "terrain": -0.05},
    "vessel":     {},              # ships are not matched (resolution_status not_geographic)
}

# Prominence (CGNDB "relevance at scale": the map scale at which a name is shown; larger = more prominent)
RELEVANCE_WEIGHT = 0.15            # bonus for the most prominent names (30,000,000), scaled logarithmically from 50,000
MIN_RELEVANCE_LOW_CONFIDENCE = 250_000   # CGNDB features less prominent than this are ignored when type_confidence is low

# Book context (section 5)
PROVINCE_CONTEXT_WEIGHT = 0.20     # bonus = weight x share of the book's anchor mentions in the candidate's province
ANCHOR_MIN_RELEVANCE    = 1_000_000
BOOK_PROVINCE_HINTS     = {        # manual additions: share 1.0 for these provinces
    "BUoL": ["saskatchewan"],      # Batoche 1885; its anchors lean to Ontario because Toronto and Ottawa are often mentioned
}
# Obscure CGNDB features outside the book's provinces are almost always namesakes ("Stanley Park", Ontario,
# in a book set in Vancouver). Once a book has context, CGNDB candidates in a province whose share is below
# max(OUTSIDE_CONTEXT_SHARE, OUTSIDE_CONTEXT_RELATIVE x the book's top share) must be at least
# OUTSIDE_CONTEXT_MIN_RELEVANCE prominent.
REJECT_OBSCURE_OUTSIDE_CONTEXT = True
OUTSIDE_CONTEXT_SHARE          = 0.05
OUTSIDE_CONTEXT_RELATIVE       = 0.5
OUTSIDE_CONTEXT_MIN_RELEVANCE  = 1_000_000

# Canada vs the world list: a name in both (London, Paris, Dublin, Liverpool) resolves to the world entry
# unless the Canadian candidate is prominent and lies in a province the book is clearly set in
CANADA_WINS_CONTEXT_SHARE = 0.5
CANADA_WINS_MIN_RELEVANCE = 1_000_000

# Names that are only a generic word ("the Fort", "the Forks", "the Heights") are story locations;
# they are not matched (status unmatched, review reason generic_name)
GENERIC_NAMES = {"fort", "forks", "heights", "fortress", "orchard", "farm", "camp", "town", "city", "village",
                 "river", "lake", "bay", "island", "harbour", "harbor", "station", "church", "school", "mission",
                 "settlement", "colony", "province", "mill", "house", "hall", "hospital", "barracks", "shed", "sheds",
                 "wharf", "pier", "point", "falls", "creek", "post", "citadel", "convent", "hill", "valley", "woods"}

AMBIGUITY_MARGIN      = 0.10       # a competitor within this score margin ...
AMBIGUITY_KM          = 25.0       # ... and farther than this distance marks a match ambiguous
MAX_CANDIDATES_LOGGED = 5          # per query and gazetteer, in place_candidates.csv

# Previous final outputs are copied to ARCHIVE_DIR before being overwritten
ARCHIVE_PREVIOUS_OUTPUTS = True
ARCHIVE_KEEP = 3

# Website
PUBLISH_TO_WEB = True              # update Web/data_sources.json and run Web/scripts/sync_data.py (section 8)

# Province normalisation (CGNDB values are names; SGC codes and abbreviations are also accepted)
PROVINCE_CODES = {
    "10": "newfoundland and labrador", "11": "prince edward island", "12": "nova scotia",
    "13": "new brunswick", "24": "quebec", "35": "ontario", "46": "manitoba",
    "47": "saskatchewan", "48": "alberta", "59": "british columbia", "60": "yukon",
    "61": "northwest territories", "62": "nunavut",
}
PROVINCE_ALIASES = {
    "nl": "newfoundland and labrador", "newfoundland": "newfoundland and labrador",
    "pe": "prince edward island", "pei": "prince edward island",
    "ns": "nova scotia", "nb": "new brunswick",
    "qc": "quebec", "que": "quebec", "pq": "quebec",
    "on": "ontario", "ont": "ontario", "mb": "manitoba", "man": "manitoba",
    "sk": "saskatchewan", "sask": "saskatchewan", "ab": "alberta", "alta": "alberta",
    "bc": "british columbia", "yt": "yukon", "yuk": "yukon",
    "nt": "northwest territories", "nwt": "northwest territories", "nu": "nunavut",
}
PROVINCE_DISPLAY = {p: p.title().replace(" And ", " and ") for p in PROVINCE_CODES.values()}

# CGNDB: column headers (matched after lower-casing and removing punctuation) and feature classes
CGNDB_COLUMN_ALIASES = {
    "cgndb_id":   ["cgndbid"],
    "name":       ["geographicalname", "geoname"],
    "generic":    ["genericterm", "generic"],
    "category":   ["genericcategory", "category"],
    "concise":    ["concisecode", "concise"],
    "feature_id": ["toponymicfeatureid", "featureid"],
    "lat":        ["latitude", "lat"],
    "lon":        ["longitude", "lon", "long"],
    "location":   ["location"],
    "province":   ["provinceterritory", "provterr", "province"],
    "relevance":  ["relevanceatscale", "relscale"],
}
CONCISE_CLASS = {                  # CGNDB concise code -> feature class; None = decided by the generic category
    "PROV": "admin_large", "TERR": "admin_large",
    "CITY": "settlement", "TOWN": "settlement", "VILG": "settlement", "HAM": "settlement",
    "UNP": None, "MUN1": None,     # populated place or administrative area
    "MUN2": "area", "GEOG": "area", "IR": "area", "PARK": "area",
    "LAKE": "water", "RIV": "water", "RIVF": "water", "BAY": "water", "CHAN": "water", "SEA": "water",
    "SEAF": "water", "SEAU": "water", "RAP": "water", "FALL": "water", "SPRG": "water", "SHL": "water", "RES": "water",
    "ISL": "island",
    "MTN": "terrain", "CAPE": "terrain", "VALL": "terrain", "CLF": "terrain", "PLN": "terrain", "GLAC": "terrain",
    "BCH": "terrain", "VEGL": "terrain", "FOR": "terrain", "CAVE": "terrain", "CRAT": "terrain",
    "AIR": "facility", "MAR": "facility", "RAIL": "facility", "ROAD": "facility", "CAMP": "facility",
    "MIL": "facility", "SITE": "facility", "RECR": "facility", "HYDR": "facility", "MISC": "other",
}
CATEGORY_CLASS = {                 # CGNDB generic category (normalised) -> feature class
    "populatedplace": "settlement", "administrativearea": "area", "waterfeature": "water",
    "terrainfeature": "terrain", "constructedfeature": "facility", "underseaandmaritimefeature": "water",
}
CLASS_PRECISION = {                # feature class -> location_precision
    "settlement": "settlement", "csd": "settlement", "facility": "site", "area": "region",
    "admin_large": "region", "country": "country", "water": "feature", "island": "feature",
    "terrain": "feature", "other": "feature",
}

# CHGIS: layer discovery, field names and linkage
CHGIS_LAYER_INCLUDE = None         # substring a layer name must contain (e.g. "CSD"); None = no filter
CHGIS_FIELDS = {                   # pin field names here if auto-detection fails; None = auto-detect
    "id": None, "name": None,
    "extra": [],                   # additional fields carried to the output as csd_<field>
}
CHGIS_FIELD_CANDIDATES = {
    "id":   ["ccriuid", "csduid", "csdid", "csd_id", "uid", "id", "geouid"],
    "name": ["csdname", "csd_name", "name", "csd", "nom", "csdnom"],
}
CSD_SNAP_M = 3000                  # a point outside every polygon is assigned the nearest one within this distance (m)
CSD_ELIGIBLE_CLASSES = {"settlement", "area", "island", "terrain", "facility", "csd", "other"}
CSD_GEOJSON_SIMPLIFY_M = None      # simplification tolerance in metres for csd_polygons.geojson; None = none

In [3]:
# ── Imports and dependency check ──────────────────────────────────────────────
import re, json, math, bisect, difflib, shutil, subprocess, sys, unicodedata
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import pyogrio

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kwargs):
        return it

print("pandas", pd.__version__, "| geopandas", gpd.__version__, "| shapely", shapely.__version__, "| pyogrio", pyogrio.__version__)

pandas 3.0.0 | geopandas 1.2.0 | shapely 2.2.0 | pyogrio 0.13.0


## 1. Place queries

Kept mentions are grouped into *queries*: one per book, folded name and place type. *Quebec* typed as a settlement and *Quebec* typed as a region in the same book are two queries and can resolve to the city and the province.

Mentions typed `unknown` take the most common known type of the same name in the same book, so a stray untyped "Quebec" joins the book's other "Quebec" mentions. The query's `type_confidence` is the highest confidence among its mentions.

In [4]:
def fold(s) -> str:
    """Normalise a place string for comparison: accents, case, punctuation, Saint/St. Same as the extraction step."""
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c)).lower()
    s = s.replace("'", "").replace("’", "").replace("‘", "").replace(".", "")
    s = re.sub(r"[-‐‑–—/]", " ", s)
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\bsaint\b", "st", s)
    s = re.sub(r"\bsainte\b", "ste", s)
    return " ".join(s.split())

def norm_province(x):
    """Lower-case modern province or territory name, or None."""
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return None
    s = fold(str(x).split("/")[0])
    if not s:
        return None
    return PROVINCE_CODES.get(s) or PROVINCE_ALIASES.get(s) or s

def haversine_km(lat1, lon1, lat2, lon2):
    if None in (lat1, lon1, lat2, lon2):
        return None
    p1, p2 = math.radians(lat1), math.radians(lat2)
    a = math.sin((p2 - p1) / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(math.radians(lon2 - lon1) / 2) ** 2
    return 2 * 6371.0088 * math.asin(math.sqrt(a))

def _to_int(x):
    try:
        return int(float(x))
    except (TypeError, ValueError):
        return None

CONFIDENCE_RANK = {"model": 3, "high": 2, "medium": 1, "low": 0}
REQUIRED = ["mention_id", "book", "entry_id", "place", "place_key", "keep", "place_type", "type_confidence", "year"]

def load_mentions(books=None):
    df = pd.read_csv(EXTRACTED_CSV, dtype=str, keep_default_na=False, encoding="utf-8")
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        raise ValueError(f"{EXTRACTED_CSV.name}: missing columns {missing}; re-run 01_place_extraction.ipynb")
    df = df[df["keep"].str.lower() == "true"].copy()
    if books is not None:
        df = df[df["book"].isin(books)]
    df["year"] = df["year"].map(_to_int)
    known = (df[df["place_type"] != "unknown"].groupby(["book", "place_key"])["place_type"]
             .agg(lambda s: s.value_counts().index[0]))
    unknown = df["place_type"] == "unknown"
    df["query_type"] = df["place_type"]
    df.loc[unknown, "query_type"] = [known.get((b, k), "unknown") for b, k in zip(df.loc[unknown, "book"], df.loc[unknown, "place_key"])]
    df["query_id"] = df["book"] + ":" + df["place_key"] + ":" + df["query_type"]
    return df.reset_index(drop=True)

def build_queries(df):
    queries = []
    for qid, g in df.groupby("query_id", sort=True):
        years = [y for y in g["year"] if y is not None]
        conf = max(g["type_confidence"], key=lambda c: CONFIDENCE_RANK.get(c, 0))
        surfaces = g["place"].value_counts()
        queries.append(dict(
            query_id=qid, book=g["book"].iat[0], place_key=g["place_key"].iat[0], name=surfaces.index[0],
            aliases=[a for a in surfaces.index[1:] if fold(a) != fold(surfaces.index[0])],
            place_type=g["query_type"].iat[0], type_confidence=conf,
            year_min=min(years) if years else None, year_max=max(years) if years else None,
            year_source=("entries" if (g.get("year_inferred", pd.Series(["True"])).str.lower() == "false").any()
                         else "default_year") if years else "unknown",
            n_mentions=len(g)))
    return queries

MENTIONS = load_mentions(BOOKS)
QUERIES = build_queries(MENTIONS)
print(f"{len(MENTIONS)} kept mentions -> {len(QUERIES)} queries in {MENTIONS['book'].nunique()} books")
pd.DataFrame(QUERIES).pivot_table(index="place_type", columns="type_confidence", values="query_id",
                                  aggfunc="count", fill_value=0, margins=True)

4384 kept mentions -> 1302 queries in 20 books


type_confidence,high,low,medium,All
place_type,,,,
country,105,0,17,122
facility,202,192,0,394
landform,44,0,0,44
region,84,0,24,108
settlement,22,371,33,426
unknown,0,95,0,95
vessel,8,0,0,8
water,101,0,4,105
All,566,658,78,1302


## 2. Name matching

Each query is expanded into name *variants*: the name, its other spellings in the book, and normalised forms. A variant carries a penalty subtracted from the name score of any match made through it.

Lookup is exact on the folded name. If no variant has an exact hit and fuzzy matching is allowed for the query (`FUZZY_CONFIDENCES`, `FUZZY_MIN_LEN`), names with a similarity ratio of at least `MIN_NAME_SCORE` are searched.

> **[DESIGN NOTE — delete before publishing]**
> **Fuzzy search only when the type is known.** Fuzzy matching against the 366,000 CGNDB names finds a near-namesake for almost any word. When the extraction step typed a name only from its NER label (`low` confidence), an exact name is required; that is also where person names that slipped through the filters end up.
>
> **Fuzzy search is blocked by first character and length.** Comparing a query with every CGNDB name using `difflib` is too slow. Names are grouped by first character and sorted by length; only names whose length can reach the threshold are compared, after the cheap `real_quick_ratio` and `quick_ratio` bounds. Cost: a misspelt *first* letter is never fuzzy-matched.

In [5]:
@dataclass
class Variant:
    text: str       # folded
    via: str        # canonical | alias | variant
    penalty: float

_LEADING_ARTICLE = re.compile(r"^(?:the|a|an)\s+")

def build_variants(query):
    sources = [(query["name"], "canonical", 0.0)] + [(a, "alias", 0.0) for a in query["aliases"]]
    out = {}

    def add(text, via, penalty):
        if text and (text not in out or penalty < out[text].penalty):
            out[text] = Variant(text, via, penalty)

    for surface, via, base in sources:
        f = fold(surface)
        add(f, via, base)
        stripped = _LEADING_ARTICLE.sub("", f)
        if stripped != f:
            add(stripped, "variant", base + ARTICLE_PENALTY)
        for lead in VARIANT_LEADING:
            if stripped.startswith(lead + " "):
                add(stripped[len(lead) + 1:], "variant", base + VARIANT_PENALTY)
        for trail in VARIANT_TRAILING:
            if stripped.endswith(" " + trail):
                add(stripped[:-len(trail) - 1], "variant", base + VARIANT_PENALTY)
        tv = TYPE_VARIANTS.get(query["place_type"], {})
        words = stripped.split()
        if words and not set(words) & set(tv.get("prefix", []) + tv.get("suffix", [])):
            for w in tv.get("prefix", []):
                add(f"{w} {stripped}", "variant", base + VARIANT_PENALTY)
            for w in tv.get("suffix", []):
                add(f"{stripped} {w}", "variant", base + VARIANT_PENALTY)
    return list(out.values())


class NameIndex:
    """Exact and fuzzy lookup over folded names. `pairs` is an iterable of (folded_name, row_position)."""

    def __init__(self, pairs):
        self.exact = defaultdict(list)
        for name, pos in pairs:
            if name:
                self.exact[name].append(pos)
        blocks = defaultdict(list)
        for name in self.exact:
            blocks[name[0]].append((len(name), name))
        self._blocks = {k: sorted(v) for k, v in blocks.items()}
        self._lens = {k: [n for n, _ in v] for k, v in self._blocks.items()}

    def fuzzy(self, q, cutoff, limit=25):
        if len(q) < FUZZY_MIN_LEN or q[0] not in self._blocks:
            return []
        lens, block = self._lens[q[0]], self._blocks[q[0]]
        lo = int(len(q) * cutoff / (2 - cutoff))
        hi = int(len(q) * (2 - cutoff) / cutoff) + 1
        sm = difflib.SequenceMatcher(autojunk=False)
        sm.set_seq2(q)
        hits = []
        for _, cand in block[bisect.bisect_left(lens, lo):bisect.bisect_right(lens, hi)]:
            sm.set_seq1(cand)
            if sm.real_quick_ratio() < cutoff or sm.quick_ratio() < cutoff:
                continue
            r = sm.ratio()
            if r >= cutoff:
                hits.append((r, cand))
        hits.sort(reverse=True)
        return [(c, r) for r, c in hits[:limit]]


def name_hits(index, variants, fuzzy):
    """{row_position: (name_score, via)} for the variants of one query."""
    hits, any_exact = {}, False
    for v in variants:
        for pos in index.exact.get(v.text, ()):
            any_exact = True
            s = 1.0 - v.penalty
            if pos not in hits or s > hits[pos][0]:
                hits[pos] = (s, v.via)
    if not any_exact and fuzzy:
        for v in variants:
            for name, ratio in index.fuzzy(v.text, MIN_NAME_SCORE):
                s = ratio - v.penalty
                for pos in index.exact[name]:
                    if pos not in hits or s > hits[pos][0]:
                        hits[pos] = (s, "fuzzy")
    return hits

## 3. Gazetteers

A gazetteer has a `name`, a `scope` (`canada` or `world`) and a method `candidates(query, variants, fuzzy)` returning `Candidate` records. Scoring and acceptance happen outside the gazetteer (section 4) so all candidates are ranked on the same terms.

### 3.1 Feature classes

Every candidate has one normalised feature class, used for type compatibility (`TYPE_CLASSES`) and for `location_precision`: `country`, `admin_large` (province, territory, colony, state), `area` (county, township, municipality, reserve, park), `settlement`, `csd` (census subdivision), `water`, `island`, `terrain`, `facility`, `other`.

In [6]:
@dataclass
class Candidate:
    gazetteer: str
    scope: str
    time_basis: str
    feature_id: str
    name: str
    lat: float
    lon: float
    feature_id_alt: str = None
    feature_type: str = None
    feature_class: str = "other"
    province: str = None
    country: str = None
    region: str = None
    relevance: float = None
    geometry_type: str = "point"
    location_precision: str = None
    name_score: float = 0.0
    matched_via: str = "canonical"
    score: float = 0.0
    extra: dict = field(default_factory=dict)


class Gazetteer:
    name, scope, time_basis, n_records, source = "base", "canada", "unspecified", 0, ""

    def candidates(self, query, variants, fuzzy):
        raise NotImplementedError


def _norm_header(h):
    return re.sub(r"[^a-z0-9]", "", str(h).lower())

def detect_columns(headers, aliases, required):
    norm = {_norm_header(h): h for h in headers}
    found = {}
    for key, options in aliases.items():
        for o in options:
            if o in norm:
                found[key] = norm[o]
                break
    missing = [k for k in required if k not in found]
    if missing:
        raise ValueError(f"Could not find columns for {missing}. Headers present: {list(headers)}")
    return found

def _read_curated(path, required, optional):
    if not Path(path).exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8")
    df.columns = [c.strip().lower() for c in df.columns]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{Path(path).name}: missing columns {missing}")
    for c in optional:
        if c not in df.columns:
            df[c] = ""
    df["_lat"] = pd.to_numeric(df["latitude"], errors="coerce")
    df["_lon"] = pd.to_numeric(df["longitude"], errors="coerce")
    bad = df["_lat"].isna() | df["_lon"].isna()
    if bad.any():
        print(f"NOTE {Path(path).name}: {int(bad.sum())} row(s) without valid coordinates skipped")
    df = df[~bad].reset_index(drop=True)
    pairs = []
    for i, r in df.iterrows():
        pairs.append((fold(r["name"]), i))
        pairs += [(fold(a), i) for a in r["aliases"].split("|") if a.strip()]
    return df, NameIndex(pairs)

### 3.2 Overrides

Hand-curated entries, matched first and by exact folded name or alias. A row applies if its `book`, year range and `place_type` (blank = any) fit the query. Used for historical names that no modern gazetteer resolves correctly, and for names whose modern namesake is the wrong feature (CGNDB's *Ville-Marie* is a town in Témiscamingue; its *Grosse Île* entries are not the quarantine station).

In [7]:
class OverridesGazetteer(Gazetteer):
    name, scope, time_basis = "overrides", "canada", "override"

    def __init__(self, path):
        self.df, self.index = _read_curated(
            path, ["name", "latitude", "longitude"],
            ["aliases", "place_type", "book", "year_from", "year_to", "matched_name", "province", "feature_class",
             "location_precision"])
        self.n_records, self.source = len(self.df), str(path)

    def candidates(self, query, variants, fuzzy):
        y_lo, y_hi = query["year_min"], query["year_max"]
        out = {}
        for v in variants:
            for i in self.index.exact.get(v.text, ()):
                r = self.df.iloc[i]
                if r["place_type"] and r["place_type"] != query["place_type"] and query["type_confidence"] != "low":
                    continue
                if r["book"] and r["book"] != query["book"]:
                    continue
                f, t = _to_int(r["year_from"]), _to_int(r["year_to"])
                if y_lo is not None and ((f is not None and y_hi < f) or (t is not None and y_lo > t)):
                    continue
                score = 1.0 - v.penalty
                if i not in out or score > out[i].name_score:
                    fc = r["feature_class"] or "other"
                    out[i] = Candidate(
                        gazetteer=self.name, scope=self.scope, time_basis=self.time_basis,
                        feature_id=f"override:{r['name']}:{i + 2}", name=r["matched_name"] or r["name"],
                        lat=float(r["_lat"]), lon=float(r["_lon"]), feature_class=fc,
                        province=norm_province(r["province"]), country="Canada",
                        location_precision=r["location_precision"] or CLASS_PRECISION.get(fc),
                        name_score=score, matched_via=v.via)
        return list(out.values())

### 3.3 World list

Countries, regions and cities outside Canada that the diaries mention. Matched by exact folded name or alias only (no fuzzy search), so only names someone has deliberately listed can resolve outside Canada.

In [8]:
WORLD_TYPE_CLASS = {"country": "country", "region": "admin_large", "settlement": "settlement", "water": "water",
                    "landform": "island", "facility": "facility"}

class WorldGazetteer(Gazetteer):
    name, scope, time_basis = "world", "world", "curated"

    def __init__(self, path):
        self.df, self.index = _read_curated(path, ["name", "latitude", "longitude", "place_type"],
                                            ["aliases", "country", "region"])
        self.n_records, self.source = len(self.df), str(path)

    def candidates(self, query, variants, fuzzy):
        out = {}
        for v in variants:
            for i in self.index.exact.get(v.text, ()):
                score = 1.0 - v.penalty
                if i in out and out[i].name_score >= score:
                    continue
                r = self.df.iloc[i]
                fc = WORLD_TYPE_CLASS.get(r["place_type"], "other")
                out[i] = Candidate(
                    gazetteer=self.name, scope=self.scope, time_basis=self.time_basis,
                    feature_id=f"world:{r['name']}:{r['place_type']}", name=r["name"],
                    lat=float(r["_lat"]), lon=float(r["_lon"]), feature_type=r["place_type"], feature_class=fc,
                    country=r["country"] or None, region=r["region"] or None,
                    location_precision=CLASS_PRECISION.get(fc), name_score=score, matched_via=v.via)
        return list(out.values())

### 3.4 CGNDB

Canadian Geographical Names Database, read from the downloaded CSV files (national file, or one file per province; records are de-duplicated on id, name and province). Records of the same feature share a Toponymic Feature ID (a river has one record per province it crosses); after scoring, each feature keeps only its best-scoring record, so the book's own province wins.

Feature class comes from the concise code (`CONCISE_CLASS`); for `UNP` and `MUN1`, which mix populated places and administrative areas, from the generic category (`CATEGORY_CLASS`). Relevance at scale is the prominence signal used in scoring.

> **[DESIGN NOTE — delete before publishing]**
> **Offline CSV instead of the GeoNames search API.** Matching sends thousands of queries, some fuzzy. Offline files make the run deterministic, free of rate limits and reproducible from a dated download. The cost is that the CSV contains only *current official* names; formerly official names are available only through the API (`category=H`). Historical names go in the overrides or come from CHGIS.
>
> **The first prototype's `UNP` gap.** Unincorporated populated places (`UNP`, about 27,000 records, including Batoche, Jasper and York Factory) had no class before and scored as `other`. They are now settlements or areas by generic category.

In [9]:
def classify_cgndb(concise, category):
    code = str(concise).strip().upper()
    c = CONCISE_CLASS.get(code)
    if c is None:
        return CATEGORY_CLASS.get(_norm_header(category), "other")
    return c


class CGNDBGazetteer(Gazetteer):
    name, scope, time_basis = "cgndb", "canada", "modern"

    def __init__(self, directory, pattern):
        files = sorted(Path(directory).glob(pattern))
        if not files:
            raise FileNotFoundError(f"No files matching {pattern} in {directory}")
        frames, self.columns_detected = [], {}
        for f in files:
            try:
                raw = pd.read_csv(f, dtype=str, keep_default_na=False, encoding="utf-8-sig")
            except UnicodeDecodeError:
                raw = pd.read_csv(f, dtype=str, keep_default_na=False, encoding="latin-1")
            cols = detect_columns(raw.columns, CGNDB_COLUMN_ALIASES, required=["name", "lat", "lon"])
            self.columns_detected = cols
            sub = pd.DataFrame({k: raw[v] for k, v in cols.items()})
            for k in CGNDB_COLUMN_ALIASES:
                if k not in sub:
                    sub[k] = ""
            frames.append(sub)
        df = pd.concat(frames, ignore_index=True)
        if (df["cgndb_id"] != "").any():
            df = df.drop_duplicates(["cgndb_id", "name", "province"])
        df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
        df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
        df["relevance"] = pd.to_numeric(df["relevance"], errors="coerce")
        df = df.dropna(subset=["lat", "lon"]).reset_index(drop=True)

        self._id = df["cgndb_id"].tolist()
        self._uid = df["feature_id"].tolist()
        self._name = df["name"].tolist()
        self._lat, self._lon = df["lat"].tolist(), df["lon"].tolist()
        self._concise = df["concise"].tolist()
        self._generic = df["generic"].tolist()
        self._rel = [None if r != r else r for r in df["relevance"]]
        self._province = [norm_province(p) for p in df["province"]]
        self._class = [classify_cgndb(c, g) for c, g in zip(df["concise"], df["category"])]
        self.index = NameIndex(zip(df["name"].map(fold), range(len(df))))
        self.n_records = len(df)
        self.source = f"{len(files)} file(s) in {directory}"

    def candidates(self, query, variants, fuzzy):
        hits = name_hits(self.index, variants, fuzzy)
        order = sorted(hits, key=lambda p: (-hits[p][0], -(self._rel[p] or 0)))
        out = []
        for p in order[:MAX_RAW_CANDIDATES]:
            score, via = hits[p]
            fc = self._class[p]
            out.append(Candidate(
                gazetteer=self.name, scope=self.scope, time_basis=self.time_basis, feature_id=self._id[p],
                feature_id_alt=self._uid[p] or None, name=self._name[p], lat=self._lat[p], lon=self._lon[p],
                feature_type=" / ".join(x for x in (self._concise[p], self._generic[p]) if x) or None,
                feature_class=fc, province=self._province[p], country="Canada", relevance=self._rel[p],
                location_precision=CLASS_PRECISION.get(fc), name_score=score, matched_via=via))
        return out

### 3.5 CHGIS

The Canadian Historical GIS provides census subdivision (CSD) polygons for the census years 1851 to 1921, one layer per year. `CHGISLayers` discovers the layers (a four-digit census year in the layer name), detects the id and name fields, and loads layers on demand. It serves two components:

- `chgis_name` (gazetteer): matches a name against CSD names in the layer nearest to the place's year, when CGNDB has no acceptable match. The reported point is the representative point of the polygon.
- `chgis_csd` (enricher, section 6): assigns a census subdivision to Canadian places matched by another gazetteer.

For a place with year *Y*, the layer used is the census year nearest to *Y* (ties go to the earlier year), provided the distance is at most `CHGIS_MAX_GAP_YEARS`.

> **[DESIGN NOTE — delete before publishing]**
> **Nearest census year, not the latest year before the entry.** CHGIS polygons re-create the geography as published for each census. A diary dated 1847 predates every layer, and one dated 1855 sits between two. The nearest layer minimises the years over which boundaries may have changed. The gap is written to `csd_gap_years`.
>
> **Representative point, not centroid.** `representative_point()` is guaranteed to lie inside the polygon; a centroid can fall outside a crescent- or coast-shaped subdivision.
>
> **Field names are detected, not assumed.** The CHGIS field names could not be verified when this notebook was written. `describe_chgis()` lists layers and fields; pin the names in `CHGIS_FIELDS` if detection picks the wrong ones.

In [10]:
class CHGISLayers:
    def __init__(self, path):
        if path is None or not Path(path).exists():
            raise FileNotFoundError(path or f"no {CHGIS_GLOBS} in {CHGIS_DIR}")
        self.path = str(path)
        listing = pyogrio.list_layers(self.path)
        self.layer_names, self._cache, self._index_cache, self._tf_cache = {}, {}, {}, {}
        lo, hi = CHGIS_CENSUS_RANGE
        for name, _geom in listing:
            if CHGIS_LAYER_INCLUDE and CHGIS_LAYER_INCLUDE not in name:
                continue
            m = re.search(r"(?<!\d)(1[89]\d{2}|19[0-2]\d)(?!\d)", name)
            if not m or not lo <= int(m.group(1)) <= hi:
                continue
            year = int(m.group(1))
            if year in self.layer_names:
                raise ValueError(f"Layers {self.layer_names[year]!r} and {name!r} both match census year {year}; "
                                 f"set CHGIS_LAYER_INCLUDE to select one kind of layer.")
            self.layer_names[year] = name
        if not self.layer_names:
            raise ValueError(f"No layers with a census year in {Path(path).name}. Layers present: {[n for n, _ in listing]}")

    @property
    def years(self):
        return sorted(self.layer_names)

    def pick(self, year):
        """(layer_year, gap) for the layer nearest to `year`; layer_year is None if the gap exceeds the maximum."""
        if year is None:
            return None, None
        layer_year = min(self.years, key=lambda y: (abs(y - year), y))
        gap = abs(layer_year - year)
        return (layer_year, gap) if gap <= CHGIS_MAX_GAP_YEARS else (None, gap)

    def get(self, year):
        if year not in self._cache:
            self._cache[year] = self._prepare(year)
        return self._cache[year]

    def extra_columns(self):
        return [f"csd_{_norm_header(f)}" for f in CHGIS_FIELDS["extra"]]

    def _prepare(self, year):
        gdf = gpd.read_file(self.path, layer=self.layer_names[year])
        if gdf.crs is None:
            raise ValueError(f"Layer {self.layer_names[year]!r} has no CRS")
        by_norm = {_norm_header(c): c for c in gdf.columns}

        def pick_field(key):
            if CHGIS_FIELDS.get(key):
                if CHGIS_FIELDS[key] not in gdf.columns:
                    raise ValueError(f"CHGIS_FIELDS[{key!r}]={CHGIS_FIELDS[key]!r} not in layer {self.layer_names[year]!r}; "
                                     f"fields: {list(gdf.columns)}")
                return CHGIS_FIELDS[key]
            for cand in CHGIS_FIELD_CANDIDATES[key]:
                if cand in by_norm:
                    return by_norm[cand]
            return None

        id_col, name_col = pick_field("id"), pick_field("name")
        if name_col is None:
            raise ValueError(f"No CSD name field detected in layer {self.layer_names[year]!r}. Fields: {list(gdf.columns)}. "
                             f"Set CHGIS_FIELDS['name'].")
        out = gpd.GeoDataFrame({
            "csd_id": gdf[id_col].astype(str) if id_col else [f"{year}:{i}" for i in range(len(gdf))],
            "csd_name": gdf[name_col].astype(str),
        }, geometry=gdf.geometry, crs=gdf.crs)
        for f, col in zip(CHGIS_FIELDS["extra"], self.extra_columns()):
            out[col] = gdf[f].astype(str) if f in gdf.columns else None
        return out[out.geometry.notna() & ~out.geometry.is_empty].reset_index(drop=True)

    def name_index(self, year):
        if year not in self._index_cache:
            layer = self.get(year)
            pairs = []
            for i, n in enumerate(layer["csd_name"]):
                pairs.append((fold(n), i))
                bare = fold(re.sub(r"\([^)]*\)", "", n))
                if bare != fold(n):
                    pairs.append((bare, i))
            self._index_cache[year] = NameIndex(pairs)
        return self._index_cache[year]

    def to_wgs84(self, year, x, y):
        if year not in self._tf_cache:
            from pyproj import Transformer
            self._tf_cache[year] = Transformer.from_crs(self.get(year).crs, 4326, always_xy=True)
        lon, lat = self._tf_cache[year].transform(x, y)
        return lat, lon


class CHGISGazetteer(Gazetteer):
    name, scope, time_basis = "chgis_name", "canada", "census"

    def __init__(self, layers):
        self.layers = layers
        self.n_records = len(layers.layer_names)
        self.source = f"{layers.path} (layers: {layers.years})"

    def candidates(self, query, variants, fuzzy):
        if query["year_min"] is None:
            return []
        layer_year, gap = self.layers.pick(round((query["year_min"] + query["year_max"]) / 2))
        if layer_year is None:
            return []
        layer, index = self.layers.get(layer_year), self.layers.name_index(layer_year)
        hits = name_hits(index, variants, fuzzy)
        extra_cols = self.layers.extra_columns()
        out = []
        for p in sorted(hits, key=lambda p: -hits[p][0])[:MAX_RAW_CANDIDATES]:
            score, via = hits[p]
            row = layer.iloc[p]
            pt = row.geometry.representative_point()
            lat, lon = self.layers.to_wgs84(layer_year, pt.x, pt.y)
            out.append(Candidate(
                gazetteer=self.name, scope=self.scope, time_basis=f"census_{layer_year}", feature_id=row["csd_id"],
                name=row["csd_name"], lat=lat, lon=lon, feature_type="census subdivision", feature_class="csd",
                country="Canada", geometry_type="polygon_point", location_precision="settlement",
                name_score=score, matched_via=via,
                extra=dict(layer_year=layer_year, gap=gap, csd_id=row["csd_id"], csd_name=row["csd_name"],
                           fields={c: row[c] for c in extra_cols})))
        return out


def describe_chgis(layers):
    """List the layers and fields detected in the CHGIS file."""
    rows = []
    for y in layers.years:
        info = pyogrio.read_info(layers.path, layer=layers.layer_names[y])
        rows.append(dict(census_year=y, layer=layers.layer_names[y], features=info["features"],
                         geometry=info["geometry_type"], fields=", ".join(info["fields"])))
    return pd.DataFrame(rows)

### 3.6 Load gazetteers

In [11]:
def build_gazetteers():
    gz, notes, chgis = {}, [], None

    def attempt(label, factory):
        try:
            g = factory()
            gz[g.name] = g
        except FileNotFoundError as e:
            notes.append(f"{label}: not available ({e})")

    attempt("overrides", lambda: OverridesGazetteer(OVERRIDES_CSV))
    attempt("world", lambda: WorldGazetteer(WORLD_CSV))
    attempt("cgndb", lambda: CGNDBGazetteer(CGNDB_DIR, CGNDB_GLOB))
    try:
        chgis = CHGISLayers(CHGIS_PATH)
        gz["chgis_name"] = CHGISGazetteer(chgis)
    except FileNotFoundError as e:
        notes.append(f"chgis: not available ({e})")
    return gz, notes, chgis

GAZETTEERS, GAZETTEER_NOTES, CHGIS = build_gazetteers()
for n in GAZETTEER_NOTES:
    print("NOTE", n)
if "cgndb" not in GAZETTEERS:
    print("WARNING: CGNDB not loaded; only overrides, the world list and CHGIS can match Canadian places.")
if not GAZETTEERS:
    raise RuntimeError("No gazetteer is available; check the gazetteer paths.")
if "cgndb" in GAZETTEERS:
    print("CGNDB columns detected:", GAZETTEERS["cgndb"].columns_detected)
pd.DataFrame([dict(gazetteer=g.name, scope=g.scope, time_basis=g.time_basis, records=g.n_records, source=g.source)
              for g in GAZETTEERS.values()])

NOTE chgis: not available (no ['*.gdb', '*.gpkg'] in /Users/sophiebird/Downloads/Dear Canada/data/Gazetteers/chgis)
CGNDB columns detected: {'cgndb_id': 'CGNDB ID', 'name': 'Geographical Name', 'generic': 'Generic Term', 'category': 'Generic Category', 'concise': 'Concise Code', 'feature_id': 'Toponymic Feature ID', 'lat': 'Latitude', 'lon': 'Longitude', 'location': 'Location', 'province': 'Province - Territory', 'relevance': 'Relevance at Scale'}


,gazetteer,scope,time_basis,records,source
0,overrides,canada,override,53,/Users/sophiebird/Downloads/Dear Canada/data/G...
1,world,world,curated,116,/Users/sophiebird/Downloads/Dear Canada/data/G...
2,cgndb,canada,modern,366235,1 file(s) in /Users/sophiebird/Downloads/Dear ...


In [12]:
# CHGIS layers and fields (use to pin CHGIS_FIELDS if detection is wrong)
describe_chgis(CHGIS) if CHGIS is not None else "CHGIS not loaded"

'CHGIS not loaded'

## 4. Matching rules

For each query:

1. **Vessels** are not matched (`not_geographic`).
2. **Overrides**: if any row applies, the best one is the match. After this step, names that are only a generic word (`GENERIC_NAMES`: "the Fort", "the Forks") are left unmatched as story locations.
3. **Canadian candidates**: CGNDB, or CHGIS names if CGNDB has none.
4. **World candidates**: the world list.
5. If both 3 and 4 have candidates, the world entry wins unless the best Canadian candidate is an exact name, at least `CANADA_WINS_MIN_RELEVANCE` prominent, and in a province with at least `CANADA_WINS_CONTEXT_SHARE` of the book's context (section 5). Either way the match is flagged `ambiguous` with the other as runner-up.

A candidate is **accepted** if
- its name score is at least `MIN_NAME_SCORE`;
- its feature class is allowed for the query's type (`TYPE_CLASSES`; overrides are exempt, they carry their own `place_type`);
- for `low` confidence queries, its CGNDB relevance is at least `MIN_RELEVANCE_LOW_CONFIDENCE`;
- once the book's context is known (section 5), a CGNDB feature in a minor province of the book (share below `OUTSIDE_CONTEXT_RELATIVE` of the top province's, or below `OUTSIDE_CONTEXT_SHARE`) is prominent (`OUTSIDE_CONTEXT_MIN_RELEVANCE`).

**Score** = name score + type bonus (`TYPE_CLASSES`) + prominence bonus (`RELEVANCE_WEIGHT`, CGNDB) + book context bonus (`PROVINCE_CONTEXT_WEIGHT` × the province's share in the book). The top-scoring accepted candidate is the match. It is **ambiguous** if a different feature farther than `AMBIGUITY_KM` scores within `AMBIGUITY_MARGIN`.

> **[DESIGN NOTE — delete before publishing]**
> **The type is a filter, not just a bonus.** The extraction step's type is the main fix for city-vs-province errors: a query typed `region` cannot match a city, one typed `water` cannot match a town. For `low` confidence types (typed only by NER label) the filter is relaxed to the `unknown` classes, and a minimum prominence applies instead, because a weakly typed name that only matches a tiny CGNDB feature is more likely a person or a story place.
>
> **Why the world list wins by default.** In the prototype, Canada-first search sent London, England to London, Ontario in every book. In these diaries a bare "London", "Paris" or "Liverpool" usually means the European city; the Canadian one is chosen only when the book is clearly set in its province.
>
> **Weights are guesses.** They are additive and small so that no single signal dominates an exact-name difference. Tune them against `place_candidates.csv`.

In [13]:
CA_RANK = ["cgndb", "chgis_name"]

def allowed_classes(q):
    allowed = dict(TYPE_CLASSES.get(q["place_type"], {}))
    if q["type_confidence"] == "low" and q["place_type"] != "vessel":
        for c, b in TYPE_CLASSES["unknown"].items():
            allowed[c] = max(b, allowed.get(c, b))
    return allowed

def relevance_bonus(rel):
    if not rel or rel <= 0:
        return 0.0
    return RELEVANCE_WEIGHT * min(1.0, max(0.0, math.log10(rel / 50_000) / math.log10(30_000_000 / 50_000)))

def accept_and_score(cands, q, allowed, context):
    best = {}
    for c in cands:
        if c.name_score < MIN_NAME_SCORE:
            continue
        if c.feature_class not in allowed and c.gazetteer != "overrides":   # overrides are filtered by their own place_type
            continue
        if c.gazetteer == "cgndb":
            rel = c.relevance or 0
            if q["type_confidence"] == "low" and rel < MIN_RELEVANCE_LOW_CONFIDENCE:
                continue
            if (REJECT_OBSCURE_OUTSIDE_CONTEXT and context and rel < OUTSIDE_CONTEXT_MIN_RELEVANCE
                    and context.get(c.province, 0.0) < max(OUTSIDE_CONTEXT_SHARE,
                                                           OUTSIDE_CONTEXT_RELATIVE * max(context.values()))):
                continue
        c.score = (c.name_score + allowed.get(c.feature_class, 0.0) + relevance_bonus(c.relevance)
                   + PROVINCE_CONTEXT_WEIGHT * context.get(c.province, 0.0))
        key = (c.gazetteer, c.feature_id_alt or c.feature_id)
        if key not in best or c.score > best[key].score:
            best[key] = c
    return sorted(best.values(), key=lambda c: (-c.score, str(c.feature_id)))

def resolve_one(q, context):
    """(status, winner, runner_up, ambiguous, considered). status: matched | unmatched | not_geographic."""
    considered = []
    if q["place_type"] == "vessel":
        return "not_geographic", None, None, False, considered
    variants = build_variants(q)
    fuzzy = FUZZY_ENABLED and q["type_confidence"] in FUZZY_CONFIDENCES
    allowed = allowed_classes(q)

    def gather(name):
        if name not in GAZETTEERS:
            return []
        acc = accept_and_score(GAZETTEERS[name].candidates(q, variants, fuzzy), q, allowed, context)
        considered.append((name, acc))
        return acc

    ov = gather("overrides")
    if ov:
        return ("matched",) + _pick(ov) + (considered,)
    if q["place_key"] in GENERIC_NAMES:            # after overrides, so a book-specific row ("the Forks") still applies
        return "unmatched", None, None, False, considered
    canada = []
    for name in CA_RANK:
        canada = gather(name)
        if canada:
            break
    world = gather("world")
    if canada and world:
        best = canada[0]
        canada_wins = (best.name_score >= 0.999 and (best.relevance or 0) >= CANADA_WINS_MIN_RELEVANCE
                       and context.get(best.province, 0.0) >= CANADA_WINS_CONTEXT_SHARE
                       and q["place_type"] != "country")
        win, other = (canada, world) if canada_wins else (world, canada)
        winner, _, _ = _pick(win)
        return "matched", winner, other[0], True, considered
    if canada or world:
        return ("matched",) + _pick(canada or world) + (considered,)
    return "unmatched", None, None, False, considered

def _pick(accepted):
    """(winner, runner_up, ambiguous) within one gazetteer's accepted candidates."""
    best = accepted[0]
    for c in accepted[1:]:
        same = c.feature_id == best.feature_id or (c.feature_id_alt and c.feature_id_alt == best.feature_id_alt)
        dist = haversine_km(best.lat, best.lon, c.lat, c.lon)
        if same or (dist is not None and dist <= AMBIGUITY_KM):
            continue
        return best, c, best.score - c.score <= AMBIGUITY_MARGIN
    return best, None, False

## 5. Book context

Which provinces is a book set in? Matching runs twice. The first pass uses only `BOOK_PROVINCE_HINTS`. Its **anchor** matches are exact-name, unambiguous Canadian matches that are either overrides, CHGIS names, or CGNDB features at least `ANCHOR_MIN_RELEVANCE` prominent. Each book's context is the share of its anchor *mentions* per province, so the places a diarist writes about most weigh most. The second pass uses that context in scoring and in the Canada-vs-world rule.

The context table is written to `intermediate/matching/book_context.csv`. If a book's context looks wrong, set `BOOK_PROVINCE_HINTS` for it.

> **[DESIGN NOTE — delete before publishing]**
> **Why computed context instead of a hand-written table only.** A table of settings per book is quick to write but easy to get subtly wrong (books move: *A Sea of Sorrows* travels from Ireland through Grosse Île to Montreal and Bytown). The anchors are the book's own unambiguous, prominent places, so the context follows what the text says. The manual hints remain for the cases where the anchors are thin.

In [14]:
def run_pass(queries, contexts):
    return {q["query_id"]: resolve_one(q, contexts.get(q["book"], {})) for q in tqdm(queries, desc="Matching")}

def manual_context(book):
    return {norm_province(p): 1.0 for p in BOOK_PROVINCE_HINTS.get(book, [])}

def compute_contexts(queries, results):
    weight = defaultdict(Counter)
    for q in queries:
        status, win, _, ambiguous, _ = results[q["query_id"]]
        if (status != "matched" or ambiguous or win.scope != "canada" or not win.province
                or win.name_score < 0.999
                or (win.gazetteer == "cgndb" and (win.relevance or 0) < ANCHOR_MIN_RELEVANCE)):
            continue
        weight[q["book"]][win.province] += q["n_mentions"]
    contexts = {}
    for book in {q["book"] for q in queries}:
        total = sum(weight[book].values())
        ctx = {p: n / total for p, n in weight[book].items()} if total else {}
        ctx.update(manual_context(book))
        contexts[book] = ctx
    return contexts, weight

first = run_pass(QUERIES, {b: manual_context(b) for b in {q["book"] for q in QUERIES}})
CONTEXTS, ANCHOR_WEIGHT = compute_contexts(QUERIES, first)
RESULTS = run_pass(QUERIES, CONTEXTS)

book_context = pd.DataFrame([dict(book=b, province=p, share=round(s, 3), anchor_mentions=ANCHOR_WEIGHT[b].get(p, 0),
                                  manual=p in manual_context(b))
                             for b, ctx in CONTEXTS.items() for p, s in ctx.items()]
                            ).sort_values(["book", "share"], ascending=[True, False])
book_context.to_csv(BOOK_CONTEXT_CSV, index=False, encoding="utf-8")
print("Top province per book")
book_context.groupby("book").head(2)

Matching:   0%|          | 0/1302 [00:00<?, ?it/s]

Matching:   0%|          | 0/1302 [00:00<?, ?it/s]

Top province per book


,book,province,share,anchor_mentions,manual
40,AToBD,british columbia,0.487,37,False
44,AToBD,manitoba,0.197,15,False
5,AinUTL,quebec,0.980,97,False
6,AinUTL,newfoundland and labrador,0.010,1,False
52,BFfH,ontario,0.837,36,False
53,BFfH,nova scotia,0.070,3,False
88,BUoL,saskatchewan,1.000,91,True
93,BUoL,ontario,0.187,29,False
72,CofOO,ontario,0.428,74,False
71,CofOO,quebec,0.283,49,False


## 6. Result rows and census subdivisions

In [15]:
def make_row(q, result):
    status, win, runner, ambiguous, _ = result
    row = {
        "query_id": q["query_id"], "book": q["book"], "name": q["name"], "aliases": "|".join(q["aliases"]),
        "place_key": q["place_key"], "place_type": q["place_type"], "type_confidence": q["type_confidence"],
        "year_min": q["year_min"], "year_max": q["year_max"], "year_source": q["year_source"],
        "n_mentions": q["n_mentions"], "status": status, "scope": None, "gazetteer": None, "time_basis": None,
        "feature_id": None, "feature_id_alt": None, "matched_name": None, "matched_via": None, "feature_type": None,
        "feature_class": None, "location_precision": None, "province": None, "country": None, "region": None,
        "latitude": None, "longitude": None, "geometry_type": None, "relevance": None, "name_score": None,
        "score": None, "ambiguous": ambiguous, "runner_up_name": None, "runner_up_gazetteer": None,
        "runner_up_distance_km": None, "book_context_share": None, "_extra": {},
    }
    reasons = []
    if status == "unmatched":
        reasons.append("generic_name" if q["place_key"] in GENERIC_NAMES else "unmatched")
    if win is not None:
        row.update(scope=win.scope, gazetteer=win.gazetteer, time_basis=win.time_basis, feature_id=win.feature_id,
                   feature_id_alt=win.feature_id_alt, matched_name=win.name, matched_via=win.matched_via,
                   feature_type=win.feature_type, feature_class=win.feature_class,
                   location_precision=win.location_precision, province=win.province, country=win.country,
                   region=win.region, latitude=win.lat, longitude=win.lon, geometry_type=win.geometry_type,
                   relevance=win.relevance, name_score=round(win.name_score, 3), score=round(win.score, 3),
                   book_context_share=round(CONTEXTS.get(q["book"], {}).get(win.province, 0.0), 3) if win.province else None,
                   _extra=win.extra)
        if runner is not None:
            row.update(runner_up_name=f"{runner.name} ({PROVINCE_DISPLAY.get(runner.province) or runner.country or ''})",
                       runner_up_gazetteer=runner.gazetteer,
                       runner_up_distance_km=round(haversine_km(win.lat, win.lon, runner.lat, runner.lon), 1))
        if ambiguous:
            reasons.append("ambiguous")
        if win.matched_via == "fuzzy":
            reasons.append("fuzzy_name")
        elif win.matched_via == "variant":
            reasons.append("variant_match")
        if q["type_confidence"] == "low":
            reasons.append("low_type_confidence")
    if q["year_source"] == "unknown":
        reasons.append("year_unknown")
    row["_reasons"] = reasons
    return row

`chgis_csd` assigns a census subdivision from the layer nearest to the place's year to every Canadian match whose feature class is in `CSD_ELIGIBLE_CLASSES`:

- Matches made by `chgis_name` take the subdivision they matched (`csd_method = name`).
- Other points are tested against the layer polygons (`point_in_polygon`); a point outside every polygon is assigned the nearest polygon within `CSD_SNAP_M` metres (`nearest`), otherwise `no_polygon`.
- Provinces, countries, water features and world matches are `not_applicable`.

In [16]:
class CHGISContainment:
    name = "chgis_csd"

    def __init__(self, layers):
        self.layers = layers

    def _blank(self):
        base = {"csd_status": "not_applicable", "csd_method": None, "csd_layer_year": None, "csd_gap_years": None,
                "csd_id": None, "csd_name": None, "csd_snap_m": None}
        base.update({c: None for c in self.layers.extra_columns()})
        return base

    def apply(self, rows):
        by_layer = defaultdict(list)
        for r in rows:
            r.update(self._blank())
            if r["status"] != "matched" or r["scope"] != "canada":
                continue
            if r["gazetteer"] == "chgis_name":
                ex = r["_extra"]
                r.update(csd_status="matched", csd_method="name", csd_layer_year=ex["layer_year"],
                         csd_gap_years=ex["gap"], csd_id=ex["csd_id"], csd_name=ex["csd_name"], **ex["fields"])
                continue
            if r["feature_class"] not in CSD_ELIGIBLE_CLASSES or r["location_precision"] == "region":
                continue
            if r["year_min"] is None:
                r["csd_status"] = "no_year"
                continue
            layer_year, gap = self.layers.pick(round((r["year_min"] + r["year_max"]) / 2))
            if layer_year is None:
                r.update(csd_status="out_of_period", csd_gap_years=gap)
                continue
            r["csd_gap_years"] = gap
            by_layer[layer_year].append(r)
        for layer_year, rs in by_layer.items():
            self._link_layer(layer_year, rs)

    def _link_layer(self, layer_year, rs):
        layer = self.layers.get(layer_year)
        cols = ["csd_id", "csd_name"] + self.layers.extra_columns()
        pts = gpd.GeoDataFrame({"k": range(len(rs))},
                               geometry=gpd.points_from_xy([r["longitude"] for r in rs], [r["latitude"] for r in rs]),
                               crs=4326).to_crs(layer.crs)
        joined = gpd.sjoin(pts, layer[cols + ["geometry"]], how="left", predicate="within")
        joined = joined[~joined.index.duplicated(keep="first")]
        method = {}
        for i in joined.index[joined["csd_id"].notna()]:
            method[i] = ("point_in_polygon", None, joined.loc[i])
        miss = pts.loc[[i for i in pts.index if i not in method]]
        if len(miss) and CSD_SNAP_M and layer.crs.is_projected:
            near = gpd.sjoin_nearest(miss, layer[cols + ["geometry"]], how="left",
                                     max_distance=CSD_SNAP_M, distance_col="snap_m")
            near = near[~near.index.duplicated(keep="first")]
            for i in near.index[near["csd_id"].notna()]:
                method[i] = ("nearest", float(near.loc[i, "snap_m"]), near.loc[i])
        for i, r in enumerate(rs):
            if i in method:
                how, snap, src = method[i]
                r.update(csd_status="matched", csd_method=how, csd_layer_year=layer_year,
                         csd_id=src["csd_id"], csd_name=src["csd_name"],
                         csd_snap_m=None if snap is None else round(snap, 1),
                         **{c: src[c] for c in self.layers.extra_columns()})
            else:
                r.update(csd_status="no_polygon", csd_layer_year=layer_year)

In [17]:
ROWS = [make_row(q, RESULTS[q["query_id"]]) for q in QUERIES]
CSD_ENRICHER = CHGISContainment(CHGIS) if CHGIS is not None else None
if CSD_ENRICHER:
    CSD_ENRICHER.apply(ROWS)
else:
    print("NOTE: census subdivision linkage skipped, CHGIS not loaded")

for r in ROWS:
    reasons = list(r.pop("_reasons"))
    if r.get("csd_method") == "nearest":
        reasons.append("csd_snapped")
    if r.get("csd_status") == "no_polygon":
        reasons.append("csd_no_polygon")
    r["review_reasons"] = "|".join(reasons)
    r["needs_review"] = bool(reasons)

print(pd.Series([r["status"] for r in ROWS]).value_counts().to_string())

NOTE: census subdivision linkage skipped, CHGIS not loaded
matched           709
unmatched         585
not_geographic      8


## 7. Historical region

`region_historical` is the political unit at the mention's year, looked up in `REGION_RULES` from the match's modern province. It falls back to the modern province if the year is unknown or no rule covers it. For world matches it is the world list's `region`, else the country. Overrides that are themselves historical regions (Upper Canada, Rupert's Land) report their own name.

`REGION_RULES` maps each modern province or territory to `(first_year, last_year, historical_name)`; `None` is open-ended.

> **[DESIGN NOTE — delete before publishing]**
> **Coarse by design.** Units are assigned from the *modern* province of the matched point, so e.g. northern Ontario before 1912 is reported as Ontario's historical unit rather than Rupert's Land, Labrador's status is omitted, and Vancouver Island and the mainland colony are merged. The table is plain data for historian review.

In [18]:
_RUPERT = "Rupert's Land / North-Western Territory"

REGION_RULES = {
    "quebec": [
        (None, 1762, "New France"),
        (1763, 1790, "Province of Quebec"),
        (1791, 1840, "Lower Canada"),
        (1841, 1866, "Province of Canada (Canada East)"),
        (1867, None, "Quebec"),
    ],
    "ontario": [
        (None, 1762, "New France"),
        (1763, 1790, "Province of Quebec"),
        (1791, 1840, "Upper Canada"),
        (1841, 1866, "Province of Canada (Canada West)"),
        (1867, None, "Ontario"),
    ],
    "nova scotia": [
        (None, 1712, "Acadia"),
        (1713, None, "Nova Scotia"),
    ],
    "new brunswick": [
        (None, 1712, "Acadia"),
        (1713, 1783, "Nova Scotia"),
        (1784, None, "New Brunswick"),
    ],
    "prince edward island": [
        (None, 1758, "Île Saint-Jean (New France)"),
        (1759, 1768, "Nova Scotia (St. John's Island)"),
        (1769, 1872, "Colony of Prince Edward Island"),
        (1873, None, "Prince Edward Island"),
    ],
    "newfoundland and labrador": [
        (None, 1948, "Newfoundland"),
        (1949, None, "Newfoundland and Labrador"),
    ],
    "manitoba": [
        (None, 1869, "Rupert's Land"),
        (1870, None, "Manitoba"),
    ],
    "saskatchewan": [
        (None, 1869, _RUPERT),
        (1870, 1904, "North-West Territories"),
        (1905, None, "Saskatchewan"),
    ],
    "alberta": [
        (None, 1869, _RUPERT),
        (1870, 1904, "North-West Territories"),
        (1905, None, "Alberta"),
    ],
    "british columbia": [
        (None, 1857, "Columbia District / New Caledonia"),
        (1858, 1870, "Colony of British Columbia / Vancouver Island"),
        (1871, None, "British Columbia"),
    ],
    "yukon": [
        (None, 1869, _RUPERT),
        (1870, 1897, "North-West Territories"),
        (1898, None, "Yukon"),
    ],
    "northwest territories": [
        (None, 1869, _RUPERT),
        (1870, None, "North-West Territories"),
    ],
    "nunavut": [
        (None, 1869, _RUPERT),
        (1870, 1998, "North-West Territories"),
        (1999, None, "Nunavut"),
    ],
}
HISTORICAL_OVERRIDE_REGIONS = {"new france", "lower canada", "upper canada", "canada east", "canada west",
                               "province of canada", "acadia", "acadie", "rupert's land", "ruperts land",
                               "north west territories", "ile saint jean", "red river settlement"}

def historical_region(match, year):
    if match["status"] != "matched":
        return None
    if match["scope"] == "world":
        return match["region"] or match["country"]
    if match["gazetteer"] == "overrides" and fold(match["name"]) in {fold(n) for n in HISTORICAL_OVERRIDE_REGIONS}:
        return match["name"]
    prov = match["province"]
    if not prov:
        return match["country"]
    if year is not None:
        for first, last, name in REGION_RULES.get(prov, []):
            if (first is None or year >= first) and (last is None or year <= last):
                return name
    return PROVINCE_DISPLAY.get(prov, prov)

def modern_region(match):
    if match["status"] != "matched":
        return None
    if match["scope"] == "world":
        return match["region"] or match["country"]
    return PROVINCE_DISPLAY.get(match["province"]) or match["country"]

## 8. Write outputs and publish

In [19]:
BASE_COLUMNS = ["query_id", "book", "name", "aliases", "place_key", "place_type", "type_confidence", "year_min",
                "year_max", "year_source", "n_mentions", "status", "scope", "gazetteer", "time_basis", "feature_id",
                "feature_id_alt", "matched_name", "matched_via", "feature_type", "feature_class", "location_precision",
                "province", "country", "region", "latitude", "longitude", "geometry_type", "relevance", "name_score",
                "score", "book_context_share", "ambiguous", "runner_up_name", "runner_up_gazetteer",
                "runner_up_distance_km"]
CSD_COLUMNS = ["csd_status", "csd_method", "csd_layer_year", "csd_gap_years", "csd_id", "csd_name", "csd_snap_m"]
EXTRA_CSD = CHGIS.extra_columns() if CSD_ENRICHER else []
MATCH_COLUMNS = BASE_COLUMNS + (CSD_COLUMNS + EXTRA_CSD if CSD_ENRICHER else []) + ["needs_review", "review_reasons"]

matches = pd.DataFrame([{c: r.get(c) for c in MATCH_COLUMNS} for r in ROWS], columns=MATCH_COLUMNS)
for c in ("year_min", "year_max", "n_mentions", "csd_layer_year", "csd_gap_years"):
    if c in matches:
        matches[c] = matches[c].astype("Int64")

# Mention-level table: kept mentions joined to their query's match
STATUS_LABEL = {"overrides": "override", "cgndb": "resolved_ca", "chgis_name": "resolved_ca", "world": "resolved_world"}
match_by_id = {r["query_id"]: r for r in ROWS}
m = MENTIONS.copy()
mrec = [match_by_id[q] for q in m["query_id"]]
m["resolution_status"] = [STATUS_LABEL.get(r["gazetteer"]) if r["status"] == "matched"
                          else ("not_geographic" if r["status"] == "not_geographic" else "unresolved") for r in mrec]
m["historicity"] = [r["latitude"] is not None for r in mrec]
for col in ("gazetteer", "matched_name", "feature_type", "location_precision", "country", "latitude", "longitude",
            "csd_name", "csd_layer_year", "needs_review", "review_reasons"):
    m[col] = [r.get(col) for r in mrec]
m["region_modern"] = [modern_region(r) for r in mrec]
m["region_historical"] = [historical_region(r, y) for r, y in zip(mrec, m["year"])]
m["place_type"] = m["query_type"]
MENTION_COLUMNS = ["mention_id", "book", "chapter_id", "entry_id", "date", "date_precision", "year_inferred", "date_raw",
                   "place", "place_type", "type_confidence", "type_evidence", "ner_label", "char_start", "char_end",
                   "sentence", "historicity", "resolution_status", "gazetteer", "matched_name", "feature_type",
                   "location_precision", "region_historical", "region_modern", "country", "latitude", "longitude",
                   "csd_name", "csd_layer_year", "needs_review", "review_reasons", "query_id"]
place_mentions = m[MENTION_COLUMNS]

### 8.0 Manual corrections

Flags and corrections made on the website's map (the *Flags & fixes* page downloads them as `place_corrections.csv`) are applied to the mention table here, after matching. A row applies when its `status` is `resolved`:

- `scope`: `mention` (one `mention_id`), `book` (every mention of that folded `place` in `book`) or `all` (every book). Broader rows are applied first, so a book or mention correction wins; within a scope, older first.
- `kind` `not_a_place` marks the mention `resolution_status = not_a_place` (the site hides it); a `place_type` sets the type and `location_precision`; `latitude`/`longitude` set the location (`gazetteer = corrections`, `resolution_status = corrected`); a `correct_name` alone (e.g. *Upper Canada*) sets `matched_name`, which the site uses to fill a region.
- Open and dismissed rows are not applied. `correction_id` records the last correction applied to each mention.

The website applies the same rules to flags not yet downloaded, so the map matches this table.

In [ ]:
TYPE_PRECISION = {"settlement": "settlement", "region": "region", "country": "country", "water": "feature",
                  "landform": "feature", "facility": "site", "vessel": "site"}
SCOPE_RANK = {"all": 1, "book": 2, "mention": 3}

def apply_corrections(pm, path):
    pm = pm.copy()
    pm["correction_id"] = ""
    if not Path(path).exists():
        return pm, 0
    fx = pd.read_csv(path, dtype=str, keep_default_na=False)
    fx = fx[(fx["status"] == "resolved") & (fx["kind"] != "other")]
    fx = fx.assign(_rank=fx["scope"].map(SCOPE_RANK).fillna(0)).sort_values(["_rank", "updated"])
    folded = pm["place"].map(fold)
    for col in ("latitude", "longitude"):
        pm[col] = pm[col].astype(float)
    for col in ("historicity", "needs_review", "review_reasons", "matched_name", "place_type", "location_precision",
                "gazetteer", "resolution_status", "region_historical", "region_modern", "country", "csd_name", "feature_type"):
        pm[col] = pm[col].astype(object)  # columns that are entirely empty load as float
    touched = pd.Series(False, index=pm.index)
    for f in fx.itertuples():
        if f.scope == "mention":
            sel = pm["mention_id"] == f.mention_id
        else:
            sel = folded == fold(f.place)
            if f.scope == "book":
                sel &= pm["book"] == f.book
        if not sel.any():
            print(f"  ! correction {f.id} ({f.place!r}, {f.scope}) matches no mention")
            continue
        touched |= sel
        pm.loc[sel, "correction_id"] = f.id
        pm.loc[sel, ["needs_review", "review_reasons"]] = [False, ""]
        if f.kind == "not_a_place":
            pm.loc[sel, "resolution_status"] = "not_a_place"
            pm.loc[sel, "historicity"] = False
            pm.loc[sel, ["latitude", "longitude"]] = np.nan
            continue
        if f.place_type:
            pm.loc[sel, "place_type"] = f.place_type
            pm.loc[sel, "location_precision"] = TYPE_PRECISION.get(f.place_type, "")
        if f.latitude and f.longitude:
            pm.loc[sel, ["latitude", "longitude"]] = [float(f.latitude), float(f.longitude)]
            pm.loc[sel, "historicity"] = True
            pm.loc[sel, "matched_name"] = f.correct_name or f.place
            pm.loc[sel, ["gazetteer", "resolution_status"]] = ["corrections", "corrected"]
            pm.loc[sel, ["region_historical", "region_modern", "country", "csd_name", "feature_type"]] = ""
            pm.loc[sel, "csd_layer_year"] = pd.NA
        elif f.correct_name:
            pm.loc[sel, "matched_name"] = f.correct_name
            pm.loc[sel, ["gazetteer", "resolution_status"]] = ["corrections", "corrected"]
    return pm, int(touched.sum())

place_mentions, n_corrected = apply_corrections(place_mentions, CORRECTIONS_CSV)
print(f"Applied manual corrections to {n_corrected} mentions"
      f" ({(place_mentions['resolution_status'] == 'not_a_place').sum()} marked not a place)")

In [20]:
def archive_previous_outputs():
    existing = [p for p in (MENTIONS_CSV, MATCHES_CSV, MATCHES_GEOJSON, CSD_GEOJSON) if p.exists()]
    if not (ARCHIVE_PREVIOUS_OUTPUTS and existing):
        return
    stamp = datetime.fromtimestamp(max(p.stat().st_mtime for p in existing)).strftime("%Y-%m-%d_%H%M")
    dest = ARCHIVE_DIR / stamp
    dest.mkdir(parents=True, exist_ok=True)
    for p in existing:
        shutil.copy2(p, dest / p.name)
    print(f"Archived previous outputs -> {dest}")
    for old in sorted((d for d in ARCHIVE_DIR.iterdir() if d.is_dir()), reverse=True)[ARCHIVE_KEEP:]:
        shutil.rmtree(old)
        print(f"Removed old archive {old.name}")

archive_previous_outputs()

place_mentions.to_csv(MENTIONS_CSV, index=False, encoding="utf-8")
matches.to_csv(MATCHES_CSV, index=False, encoding="utf-8")

records = json.loads(matches.astype(object).where(matches.notna(), None).to_json(orient="records", force_ascii=False))
with_xy = [r for r in records if r["latitude"] is not None]
MATCHES_GEOJSON.write_text(json.dumps({"type": "FeatureCollection", "features": [
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": [r["longitude"], r["latitude"]]},
     "properties": {k: v for k, v in r.items() if k not in ("latitude", "longitude")}} for r in with_xy]},
    ensure_ascii=False), encoding="utf-8")

csd_frames = []
if CSD_ENRICHER:
    linked = matches[matches["csd_status"] == "matched"].dropna(subset=["csd_layer_year"])
    for layer_year, g in linked.groupby("csd_layer_year"):
        layer = CHGIS.get(int(layer_year))
        sel = layer[layer["csd_id"].isin(set(g["csd_id"]))].copy()
        if CSD_GEOJSON_SIMPLIFY_M:
            sel["geometry"] = sel.geometry.simplify(CSD_GEOJSON_SIMPLIFY_M)
        sel["layer_year"] = int(layer_year)
        sel["queries"] = sel["csd_id"].map(g.groupby("csd_id")["name"].agg(lambda s: "|".join(sorted(set(s)))))
        csd_frames.append(sel.to_crs(4326))
if csd_frames:
    pd.concat(csd_frames).pipe(gpd.GeoDataFrame, crs=4326).to_file(CSD_GEOJSON, driver="GeoJSON")
elif CSD_GEOJSON.exists():
    CSD_GEOJSON.unlink()

# Intermediate: candidate log and run metadata
log = []
for q in QUERIES:
    status, win, _, _, considered = RESULTS[q["query_id"]]
    for gname, cands in considered:
        for rank, c in enumerate(cands[:MAX_CANDIDATES_LOGGED], start=1):
            log.append({"query_id": q["query_id"], "gazetteer": gname, "rank": rank, "feature_id": c.feature_id,
                        "name": c.name, "feature_type": c.feature_type, "feature_class": c.feature_class,
                        "province": c.province, "country": c.country, "relevance": c.relevance,
                        "latitude": c.lat, "longitude": c.lon, "name_score": round(c.name_score, 3),
                        "score": round(c.score, 3), "matched_via": c.matched_via, "selected": c is win})
pd.DataFrame(log).to_csv(CANDIDATES_CSV, index=False, encoding="utf-8")

RUN_METADATA_JSON.write_text(json.dumps({
    "created": datetime.now().isoformat(timespec="seconds"),
    "input": str(EXTRACTED_CSV), "books": BOOKS,
    "parameters": dict(MIN_NAME_SCORE=MIN_NAME_SCORE, FUZZY_ENABLED=FUZZY_ENABLED, FUZZY_MIN_LEN=FUZZY_MIN_LEN,
                       FUZZY_CONFIDENCES=sorted(FUZZY_CONFIDENCES), TYPE_CLASSES=TYPE_CLASSES,
                       RELEVANCE_WEIGHT=RELEVANCE_WEIGHT, MIN_RELEVANCE_LOW_CONFIDENCE=MIN_RELEVANCE_LOW_CONFIDENCE,
                       PROVINCE_CONTEXT_WEIGHT=PROVINCE_CONTEXT_WEIGHT, ANCHOR_MIN_RELEVANCE=ANCHOR_MIN_RELEVANCE,
                       BOOK_PROVINCE_HINTS=BOOK_PROVINCE_HINTS, CANADA_WINS_CONTEXT_SHARE=CANADA_WINS_CONTEXT_SHARE,
                       CANADA_WINS_MIN_RELEVANCE=CANADA_WINS_MIN_RELEVANCE, AMBIGUITY_MARGIN=AMBIGUITY_MARGIN,
                       AMBIGUITY_KM=AMBIGUITY_KM, CHGIS_MAX_GAP_YEARS=CHGIS_MAX_GAP_YEARS, CSD_SNAP_M=CSD_SNAP_M),
    "gazetteers": {g.name: dict(records=g.n_records, source=g.source, time_basis=g.time_basis) for g in GAZETTEERS.values()},
    "notes": GAZETTEER_NOTES,
    "counts": {"mentions": len(place_mentions), "located_mentions": int(place_mentions["historicity"].sum()),
               "queries": len(matches), "by_status": matches["status"].value_counts().to_dict(),
               "needs_review": int(matches["needs_review"].sum())},
}, indent=1, default=str), encoding="utf-8")

print(f"Wrote {len(place_mentions)} mentions ({int(place_mentions['historicity'].sum())} located) -> {MENTIONS_CSV}")
print(f"Wrote {len(matches)} queries -> {MATCHES_CSV}")
print(f"Wrote {len(with_xy)} points -> {MATCHES_GEOJSON}")
print(f"Wrote {sum(len(f) for f in csd_frames)} polygons -> {CSD_GEOJSON}" if csd_frames else "No census subdivision polygons")
print(f"Wrote candidate log, book context and run metadata -> {WORK_DIR}")

Archived previous outputs -> /Users/sophiebird/Downloads/Dear Canada/data/Pipe Outputs/Archived Versions/places/2026-10-21_0020
Wrote 4384 mentions (3169 located) -> /Users/sophiebird/Downloads/Dear Canada/data/Pipe Outputs/places/place_mentions.csv
Wrote 1302 queries -> /Users/sophiebird/Downloads/Dear Canada/data/Pipe Outputs/places/place_matches.csv
Wrote 709 points -> /Users/sophiebird/Downloads/Dear Canada/data/Pipe Outputs/places/place_matches.geojson
No census subdivision polygons
Wrote candidate log, book context and run metadata -> /Users/sophiebird/Downloads/Dear Canada/Pipelines/Place Mentions/intermediate/matching


### 8.1 Publish to the website

Points the `places` project in `Web/data_sources.json` at the files above (plus the region boundaries from `03_region_boundaries.py`, the region tables and the corrections table) and runs `Web/scripts/sync_data.py`, which copies them to `Web/public/data/places/`, removes files that are no longer listed, and rebuilds the manifest. Other projects in `data_sources.json` are left as they are. Commit and push `Web/` to redeploy.

In [21]:
WEB_PLACES_DATASETS = [
    {"project": "places", "source": "../data/Pipe Outputs/places/place_mentions.csv", "role": "mentions"},
    {"project": "places", "source": "../data/Pipe Outputs/places/place_matches.csv"},
    {"project": "places", "source": "../data/Pipe Outputs/places/boundaries_hacolp.json", "role": "boundaries"},
    {"project": "places", "source": "../data/Pipe Outputs/places/boundaries_world.json", "role": "world_boundaries"},
    {"project": "places", "source": "../data/Gazetteers/curated/region_aliases.csv", "role": "region_aliases"},
    {"project": "places", "source": "../data/Gazetteers/curated/region_polygons.csv", "role": "region_polygons"},
    {"project": "places", "source": "../data/Gazetteers/curated/place_corrections.csv", "role": "corrections"},
]

def publish_to_web():
    cfg = json.loads(WEB_SOURCES_JSON.read_text(encoding="utf-8"))
    others = [d for d in cfg["datasets"] if d.get("project") != "places"]
    new = WEB_PLACES_DATASETS + others
    if new != cfg["datasets"]:
        cfg["datasets"] = new
        WEB_SOURCES_JSON.write_text(json.dumps(cfg, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
        print(f"Updated {WEB_SOURCES_JSON}")
    res = subprocess.run([sys.executable, str(WEB_SYNC_SCRIPT)], cwd=WEB_DIR, capture_output=True, text=True)
    print(res.stdout)
    if res.returncode:
        print(res.stderr)
        raise RuntimeError("sync_data.py failed")

if PUBLISH_TO_WEB:
    publish_to_web()
else:
    print("PUBLISH_TO_WEB is False; website not updated.")

  + data/places/place_mentions.csv  (4384 rows)
  + data/places/place_matches.csv  (1302 rows)
  + data/network/SofS_entries_halved_edges.json  (1450 rows)
  + data/network/SofS_entries_halved_mentions.json  (851 rows)
  + data/network/SofS_entries_halved_nodes_entries.json  (137 rows)
  + data/network/SofS_entries_halved_nodes_incidents.json  (22 rows)
  + data/network/SofS_entries_halved_nodes_stays.json  (181 rows)
  + data/network/SofS_entries_halved_nodes_characters.json  (176 rows)
  + data/network/SofS_entries_halved_nodes_dates.json  (138 rows)
  + data/network/SofS_entries_halved_nodes_health_states.json  (45 rows)
  + data/network/SofS_entries_halved_nodes_identity_categories.json  (67 rows)
  + data/network/SofS_entries_halved_nodes_locations.json  (62 rows)
  + data/network/SofS_entries_halved_nodes_organizations.json  (17 rows)
  + data/network/SofS_entries_halved_nodes_regions.json  (30 rows)
Wrote manifest with 14 datasets.



## 9. Diagnostics

The review tables are the main input for tuning the weights, the overrides and the world list.

In [22]:
print("Mentions by resolution status, per book")
print(pd.crosstab(place_mentions["book"], place_mentions["resolution_status"], margins=True).to_string())
print("\nLocated mentions by precision")
print(place_mentions.loc[place_mentions["historicity"], "location_precision"].value_counts().to_string())

Mentions by resolution status, per book
resolution_status  not_geographic  override  resolved_ca  resolved_world  unresolved   All
book                                                                                      
AToBD                           0        60           58              10          35   163
AinUTL                          0        41           68              31          23   163
BFfH                            2         4           48              39          28   121
BUoL                            0        25          151              14          35   225
CofOO                           0        88          135              34          65   322
DRtoF                           0        24           68              40          93   225
DofTT                           0         9           62               7          59   137
DofmyC                          0        21           78              19          45   163
EftW                            0        37       

In [23]:
print("Review reasons (queries)")
print(matches["review_reasons"].str.split("|").explode().replace("", np.nan).value_counts().to_string())

Review reasons (queries)
review_reasons
unmatched              569
low_type_confidence    321
ambiguous               82
generic_name            16
variant_match           14
fuzzy_name              10


In [24]:
print("Unmatched places, most frequent first (candidates for overrides, the world list, or story locations)")
matches[matches["status"] == "unmatched"].sort_values("n_mentions", ascending=False)[
    ["book", "name", "place_type", "type_confidence", "n_mentions", "year_min"]].head(40)

Unmatched places, most frequent first (candidates for overrides, the world list, or story locations)


,book,name,place_type,type_confidence,n_mentions,year_min
947,PintPL,Horoshova,settlement,low,42,1914
1175,TFN,Mill House,facility,high,28,1912
571,FitS,Fort,facility,low,26,1815
881,PasWide,Milorie,settlement,low,18,1926
261,CofOO,Parliament Buildings,facility,low,12,1866
1085,TA,Oxford Street,facility,high,12,1941
318,DRtoF,Fortress,facility,low,12,1863
971,PintPL,Spirit Lake,water,high,11,1915
862,PasWide,Halley Road,facility,high,10,1926
1048,TA,Guide Camp,facility,high,10,1941


In [25]:
print("Ambiguous matches (check the winner; add an override with a book column to force the other)")
matches[matches["ambiguous"].fillna(False)].sort_values("n_mentions", ascending=False)[
    ["book", "name", "place_type", "gazetteer", "matched_name", "province", "country", "runner_up_name",
     "runner_up_distance_km", "n_mentions"]].head(40)

Ambiguous matches (check the winner; add an override with a book column to force the other)


,book,name,place_type,gazetteer,matched_name,province,country,runner_up_name,runner_up_distance_km,n_mentions
366,DRtoF,Virginia,settlement,world,Virginia,NaN,United States,Virginia (Ontario),767.4,12
176,BUoL,Red River,water,cgndb,Red River,manitoba,Canada,Red River (Nova Scotia),2651.3,11
871,PasWide,London,settlement,world,London,NaN,United Kingdom,London (Ontario),5876.7,10
946,PintPL,Hamburg,settlement,world,Hamburg,NaN,Germany,Hamburg (Manitoba),6741.5,9
1169,TFN,London,settlement,world,London,NaN,United Kingdom,London (Ontario),5876.7,9
236,CofOO,London,settlement,world,London,NaN,United Kingdom,London (Ontario),5876.7,9
1196,TFN,Southampton,settlement,world,Southampton,NaN,United Kingdom,Southampton (Nova Scotia),4547.8,7
810,OatmyD,London,settlement,cgndb,London,ontario,Canada,London (United Kingdom),5876.7,6
1000,SofS,Liverpool,settlement,world,Liverpool,NaN,United Kingdom,Liverpool (Nova Scotia),4496.2,6
687,NSH,St. Paul,facility,cgndb,St. Paul,alberta,Canada,Saint-Paul (New Brunswick),3341.0,6


In [26]:
print("Same name, different types in one book (e.g. Quebec city vs province)")
dup = matches.groupby(["book", "place_key"])["place_type"].transform("nunique") > 1
matches[dup].sort_values(["book", "place_key"])[
    ["book", "name", "place_type", "type_confidence", "n_mentions", "matched_name", "feature_type", "province"]].head(40)

Same name, different types in one book (e.g. Quebec city vs province)


,book,name,place_type,type_confidence,n_mentions,matched_name,feature_type,province
43,AToBD,St. Anne,facility,low,2,NaN,NaN,NaN
44,AToBD,St. Anne,settlement,low,1,NaN,NaN,NaN
69,AinUTL,Montréal,facility,low,1,Montréal,CITY / Town,quebec
70,AinUTL,Montréal,settlement,low,38,Montréal,CITY / Town,quebec
77,AinUTL,Reignac,facility,low,1,NaN,NaN,NaN
78,AinUTL,Reignac,settlement,medium,5,NaN,NaN,NaN
87,BFfH,Amiens,facility,low,1,Amiens,UNP / Locality,saskatchewan
88,BFfH,Amiens,settlement,low,1,Amiens,UNP / Locality,saskatchewan
139,BUoL,Batoche,facility,low,8,Batoche,UNP / Locality,saskatchewan
140,BUoL,Batoche,settlement,low,32,Batoche,UNP / Locality,saskatchewan


In [27]:
print("Random sample of matches for spot-checking")
s = matches[matches["status"] == "matched"]
s.sample(min(25, len(s)), random_state=0)[
    ["book", "name", "place_type", "gazetteer", "matched_name", "feature_type", "province", "country",
     "score", "review_reasons"]]

Random sample of matches for spot-checking


,book,name,place_type,gazetteer,matched_name,feature_type,province,country,score,review_reasons
1006,SofS,Quebec,settlement,cgndb,Québec,CITY / Town,quebec,Canada,1.415,
873,PasWide,Lyons,facility,cgndb,Lyons,UNP / Hamlet,saskatchewan,Canada,1.213,low_type_confidence
666,NSH,Norway,country,world,Norway,country,NaN,Norway,1.100,
279,CofOO,Sandy Hill,landform,cgndb,Sandy Hill,MTN / Hill,ontario,Canada,1.186,
1178,TFN,New York,settlement,world,New York,settlement,NaN,United States,1.100,
1234,WofW,Fort George,facility,overrides,"Fort George, Niagara-on-the-Lake",NaN,ontario,Canada,1.280,
181,BUoL,Scotland,country,world,Scotland,country,NaN,United Kingdom,1.100,
58,AinUTL,Cap Diamant,landform,cgndb,Cap Diamant,CAPE / Cap,quebec,Canada,1.366,ambiguous
683,NSH,Saint John,settlement,cgndb,Saint John,CITY / City,new brunswick,Canada,1.251,
1007,SofS,Richelieu,unknown,cgndb,Richelieu,CITY / Town,quebec,Canada,1.289,low_type_confidence
